# SAMATRIX RESUMEFORGE 2026: END-TO-END NLP RESUME CLASSIFICATION
### A Complete Phase-Wise NLP Lifecycle (Total: 70 Marks)
**Workflow: Problem Definition → Data Gathering → Quality Audit → EDA → Text Preprocessing → Stratified Split → Feature Engineering → Classical & Deep Learning Models → Multi-Metric Evaluation → Error Analysis → Inference Pipeline & Testing**

This notebook fulfills all 12 stages specified in the official **Problem statement and Guidelines for Hackathon.docx** with zero data leakage, reproducible preprocessing, and comprehensive diagnostics.

In [1]:
# Setup Environment and Imports
import os
import re
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
from pathlib import Path

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configuration Constants
RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
# Support running this notebook from either the project root or notebooks/.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "Dataset").exists() else Path.cwd().parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
CACHE_PATH = ARTIFACTS_DIR / "processed_data_cache.joblib"

# Create artifact directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment configured successfully. Project root: {PROJECT_ROOT}")

Environment configured successfully. Project root: D:\ResumeForge


In [2]:
# Load the leakage-free Phase 3 feature matrices and labels.
# Run Phase 3 first if this cache is not available.
if not CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CACHE_PATH}. Run Phase3_Text_Preprocessing_and_Features.ipynb first."
    )

cache = joblib.load(CACHE_PATH)
required_keys = {"X_train", "X_val", "X_test", "y_train", "y_val", "y_test"}
missing_keys = required_keys.difference(cache)
if missing_keys:
    raise KeyError(f"Phase 3 cache is incomplete. Missing: {sorted(missing_keys)}")

X_train, X_val, X_test = cache["X_train"], cache["X_val"], cache["X_test"]
y_train, y_val, y_test = cache["y_train"], cache["y_val"], cache["y_test"]
print(f"Loaded Phase 3 matrices: train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}")

Loaded Phase 3 matrices: train=(1738, 12000), validation=(373, 12000), test=(373, 12000)


---
## 🤖 STAGE 8 & 9: Classical ML & Deep Neural Network Models (10 Marks)
We benchmark four diverse architectures:
1. **Multinomial Naive Bayes** (Fast classical text baseline)
2. **Logistic Regression** (Balanced multi-class linear classifier)
3. **Linear Support Vector Classifier** (Calibrated Linear SVM)
4. **Deep Neural Network (MLP)** (Dense network: $32 \rightarrow 16$ hidden units, ReLU, Adam, early stopping)

In [3]:
# Stage 8 & 9: Model Training and Validation Comparison
candidate_models = {
    "Multinomial_Naive_Bayes": MultinomialNB(alpha=0.1),
    "Logistic_Regression": LogisticRegression(
        C=1.0, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Linear_SVM": CalibratedClassifierCV(
        LinearSVC(C=10.0, class_weight="balanced", random_state=RANDOM_STATE, max_iter=10000),
        cv=3
    ),
    "Deep_Neural_Network_MLP": MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=0.0005,
        batch_size=64,
        learning_rate="adaptive",
        max_iter=100,
        early_stopping=True,
        random_state=RANDOM_STATE
    )
}

val_results = {}
best_model_name = None
best_macro_f1 = -1.0
best_model = None

for name, model in candidate_models.items():
    print(f"Training {name} on {X_train.shape[0]} samples...")
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred_val)
    macro_f1 = f1_score(y_val, y_pred_val, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_val, y_pred_val, average="weighted", zero_division=0)
    
    val_results[name] = {
        "Val Accuracy": f"{acc*100:.2f}%",
        "Val Macro-F1": round(macro_f1, 4),
        "Val Weighted-F1": round(weighted_f1, 4)
    }
    
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_model_name = name
        best_model = model

# Display Comparison
df_comparison = pd.DataFrame(val_results).transpose()
print("\n" + "="*60)
print(f"BEST MODEL SELECTED: {best_model_name} (Validation Macro-F1: {best_macro_f1:.4f})")
print("="*60)
display(df_comparison)

# Save best model
joblib.dump(best_model, MODELS_DIR / "best_model.joblib")
machine_readable_results = {
    name: {
        "val_accuracy": round(float(metrics["Val Accuracy"].strip("%")) / 100, 4),
        "val_macro_f1": metrics["Val Macro-F1"],
        "val_weighted_f1": metrics["Val Weighted-F1"],
    }
    for name, metrics in val_results.items()
}
with open(REPORTS_DIR / "model_comparison.json", "w", encoding="utf-8") as f:
    json.dump({"model_comparison": machine_readable_results, "best_model": best_model_name, "best_val_macro_f1": round(float(best_macro_f1), 4)}, f, indent=4)
print(f"Saved best model to {MODELS_DIR / 'best_model.joblib'}")

Training Multinomial_Naive_Bayes on 1738 samples...
Training Logistic_Regression on 1738 samples...


Training Linear_SVM on 1738 samples...


Training Deep_Neural_Network_MLP on 1738 samples...



BEST MODEL SELECTED: Linear_SVM (Validation Macro-F1: 0.6485)


,Val Accuracy,Val Macro-F1,Val Weighted-F1
Multinomial_Naive_Bayes,54.16%,0.4815,0.5202
Logistic_Regression,63.54%,0.5952,0.6224
Linear_SVM,68.36%,0.6485,0.6785
Deep_Neural_Network_MLP,51.47%,0.4839,0.5204


Saved best model to D:\ResumeForge\artifacts\models\best_model.joblib
